# Bayesian Survival Analysis with PyMC: Modelling Customer Churn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_survival_analysis.ipynb)

This notebook implements a Bayesian accelerated failure time (AFT) survival model using PyMC. We model customer churn with right-censored data, compare Weibull and Log-Logistic distributions, and plot individual survival curves with uncertainty.

**Blog post:** [Bayesian Survival Analysis with PyMC](https://sesen.ai/blog/bayesian-survival-analysis-pymc)

**Prerequisites:** [Hierarchical Bayesian Regression with PyMC](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/hierarchical_bayesian_regression.ipynb)

In [ ]:
# Install dependencies (Colab)
# !pip install pymc arviz matplotlib numpy pandas -q

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Generate Synthetic Churn Data

We simulate 1,000 customers observed over a 24-month window. Each customer has two features:
- **Monthly spend**: higher spend → longer survival (loyal customers)
- **Support tickets**: more tickets → faster churn (frustrated customers)

True churn times are generated from a Weibull distribution (via its log-time Gumbel equivalent). Customers who haven't churned by month 24 are right-censored.

In [ ]:
np.random.seed(42)

N = 1000
monthly_spend = np.random.normal(100, 30, N).clip(20, 250)
support_tickets = np.random.poisson(3, N).astype(float)

# Standardise covariates
spend_std = (monthly_spend - 100) / 30
tickets_std = (support_tickets - 3) / 2

# True AFT parameters (Gumbel / log-Weibull parameterisation)
true_alpha = np.array([2.5, 0.4, -0.3])  # intercept, spend, tickets
true_s = 0.6

# True log-time: Y = eta + s * W, where W ~ Gumbel(0, 1)
eta_true = true_alpha[0] + true_alpha[1] * spend_std + true_alpha[2] * tickets_std
log_time_true = eta_true + true_s * np.random.gumbel(0, 1, N)
time_true = np.exp(log_time_true)

# Administrative censoring at 24 months
observation_window = 24.0
observed_time = np.minimum(time_true, observation_window)
censored = time_true > observation_window  # True = still active
log_observed_time = np.log(observed_time)

print(f"Total customers: {N}")
print(f"Churned: {(~censored).sum()} ({(~censored).mean():.0%})")
print(f"Still active (censored): {censored.sum()} ({censored.mean():.0%})")

### Visualise the Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of observed times
ax = axes[0]
ax.hist(observed_time[~censored], bins=30, alpha=0.7, color='#F44336', label='Churned')
ax.hist(observed_time[censored], bins=10, alpha=0.7, color='#4CAF50', label='Censored')
ax.set_xlabel('Observed time (months)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Observed Times')
ax.legend()

# Swimmers plot (sample of 15 customers)
ax = axes[1]
np.random.seed(123)
sample = np.random.choice(N, 15, replace=False)
sample = sample[np.argsort(observed_time[sample])]
for i, idx in enumerate(sample):
    t = observed_time[idx]
    is_cens = censored[idx]
    color = '#4CAF50' if is_cens else '#F44336'
    ax.barh(i, t, height=0.6, color=color, alpha=0.7)
    marker = '→' if is_cens else '✕'
    ax.text(t + 0.3, i, marker, fontsize=10, va='center',
            color='#4CAF50' if is_cens else '#D32F2F')

ax.axvline(x=24, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Months since signup')
ax.set_ylabel('Customer')
ax.set_title('Right-Censoring: Sample of 15 Customers')
ax.set_xlim(0, 28)
plt.tight_layout()
plt.show()

## 2. Kaplan-Meier Survival Curve

The Kaplan-Meier estimator provides a non-parametric estimate of the survival function. It handles censoring correctly by adjusting the risk set at each event time.

In [ ]:
# Manual Kaplan-Meier estimator
order = np.argsort(observed_time)
times_sorted = observed_time[order]
events_sorted = (~censored)[order].astype(int)

km_times = [0.0]
km_survival = [1.0]
n_at_risk = N

for t, event in zip(times_sorted, events_sorted):
    if event:
        km_survival.append(km_survival[-1] * (1 - 1 / n_at_risk))
        km_times.append(t)
    n_at_risk -= 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(km_times, km_survival, where='post', color='#2196F3', lw=2)
ax.set_xlabel('Months since signup', fontsize=12)
ax.set_ylabel('Survival probability', fontsize=12)
ax.set_title('Kaplan-Meier Survival Curve', fontsize=14)
ax.set_xlim(0, 25)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 3. Weibull AFT Model in PyMC

The key insight: if $T \sim \text{Weibull}$, then $Y = \log T$ follows a Gumbel distribution. We model log-time with a Gumbel likelihood.

For **uncensored** customers: standard Gumbel density (via `pm.Gumbel`).

For **censored** customers: survival function (via `pm.Potential`).

$$\ell(\theta) = \sum_{i \in \text{uncensored}} \log f(t_i \mid \theta) + \sum_{i \in \text{censored}} \log S(c_i \mid \theta)$$

In [ ]:
def gumbel_log_sf(y, mu, sigma):
    """Log survival function of the Gumbel distribution."""
    return pt.log1p(-pt.exp(-pt.exp(-(y - mu) / sigma)))


with pm.Model() as weibull_aft:
    # Location coefficients (priors match original code: Normal(0, 2))
    alpha = pm.Normal('alpha', mu=0, sigma=2, shape=3)

    # Scale parameter (must be positive)
    log_s = pm.Normal('log_s', mu=0, sigma=1)
    s = pm.Deterministic('s', pm.math.exp(log_s))

    # Linear predictor for log-time
    eta = alpha[0] + alpha[1] * spend_std + alpha[2] * tickets_std

    # Uncensored customers: standard Gumbel likelihood
    y_obs = pm.Gumbel('y_obs', mu=eta[~censored], beta=s,
                       observed=log_observed_time[~censored])

    # Censored customers: survival function via pm.Potential
    y_cens = pm.Potential('y_cens',
        gumbel_log_sf(log_observed_time[censored], eta[censored], s))

    # Sample the posterior
    trace = pm.sample(1000, tune=2000, cores=4, chains=4,
                      random_seed=42, target_accept=0.9)

print(az.summary(trace, var_names=['alpha', 's']))

### MCMC Diagnostics

Check convergence: chains should mix well ("hairy caterpillars"), R-hat < 1.01, ESS > 400.

In [ ]:
az.plot_trace(trace, var_names=['alpha', 's'], compact=True)
plt.suptitle('MCMC Trace Plots: Weibull AFT Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Verify against true parameters
summary = az.summary(trace, var_names=['alpha', 's'])
print("\nTrue vs Estimated:")
print(f"  alpha[0] (intercept): true={true_alpha[0]:.2f}, est={summary.loc['alpha[0]', 'mean']:.3f}")
print(f"  alpha[1] (spend):     true={true_alpha[1]:.2f}, est={summary.loc['alpha[1]', 'mean']:.3f}")
print(f"  alpha[2] (tickets):   true={true_alpha[2]:.2f}, est={summary.loc['alpha[2]', 'mean']:.3f}")
print(f"  s (scale):            true={true_s:.2f}, est={summary.loc['s', 'mean']:.3f}")

## 4. Survival Curves from the Posterior

For any customer profile, we compute the predicted survival probability at each time point across all posterior samples. This gives us full uncertainty quantification.

In [ ]:
t_grid = np.linspace(0.5, 36, 200)
log_t_grid = np.log(t_grid)

# Extract posterior samples
alpha_post = trace.posterior['alpha'].values.reshape(-1, 3)
s_post = trace.posterior['s'].values.flatten()


def compute_survival_weibull(alpha_post, s_post, sp, tk, log_t_grid):
    """Compute Weibull survival curves from posterior samples."""
    eta_post = alpha_post[:, 0] + alpha_post[:, 1] * sp + alpha_post[:, 2] * tk
    survival = np.zeros((len(eta_post), len(log_t_grid)))
    for i in range(len(eta_post)):
        z = (log_t_grid - eta_post[i]) / s_post[i]
        survival[i] = 1 - np.exp(-np.exp(-z))
    return survival


profiles = {
    'High-value (spend +1.5σ, tickets −1σ)': (1.5, -1.0, '#2196F3'),
    'Average customer':                        (0.0,  0.0, '#FF9800'),
    'At-risk (spend −1.5σ, tickets +2σ)':     (-1.5,  2.0, '#F44336'),
}

fig, ax = plt.subplots(figsize=(8, 5))
for label, (sp, tk, color) in profiles.items():
    survival = compute_survival_weibull(alpha_post, s_post, sp, tk, log_t_grid)
    mean_surv = survival.mean(axis=0)
    lower = np.percentile(survival, 3, axis=0)
    upper = np.percentile(survival, 97, axis=0)
    ax.plot(t_grid, mean_surv, color=color, lw=2, label=label)
    ax.fill_between(t_grid, lower, upper, color=color, alpha=0.15)

ax.set_xlabel('Months since signup', fontsize=12)
ax.set_ylabel('Survival probability', fontsize=12)
ax.set_title('Predicted Survival Curves by Customer Profile', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 36)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Median Survival Times

In [ ]:
# Compute median survival time for each profile
print("Predicted median survival times (months):")
for label, (sp, tk, _) in profiles.items():
    eta_post = alpha_post[:, 0] + alpha_post[:, 1] * sp + alpha_post[:, 2] * tk
    # Median of Gumbel is mu - sigma * ln(ln(2))
    log_median = eta_post - s_post * np.log(np.log(2))
    median_months = np.exp(log_median)
    print(f"  {label}: {np.mean(median_months):.1f} months "
          f"(94% HDI: [{np.percentile(median_months, 3):.1f}, {np.percentile(median_months, 97):.1f}])")

## 5. Heteroscedastic Model (Covariates in the Scale)

The basic model uses a constant scale parameter `s`. The original code models the scale as covariate-dependent too:

$$s_i = \exp(\rho_0 + \rho_1 \cdot x_{i1} + \rho_2 \cdot x_{i2})$$

This means the *shape* of the Weibull hazard varies across customers.

In [ ]:
with pm.Model() as weibull_aft_hetero:
    # Location coefficients
    alpha_h = pm.Normal('alpha', mu=0, sigma=2, shape=3)
    # Scale coefficients (matching original code's rho priors)
    rho = pm.Normal('rho', mu=0, sigma=2, shape=3)

    eta_h = alpha_h[0] + alpha_h[1] * spend_std + alpha_h[2] * tickets_std
    s_h = pm.math.exp(rho[0] + rho[1] * spend_std + rho[2] * tickets_std)

    y_obs_h = pm.Gumbel('y_obs', mu=eta_h[~censored], beta=s_h[~censored],
                         observed=log_observed_time[~censored])
    y_cens_h = pm.Potential('y_cens',
        gumbel_log_sf(log_observed_time[censored], eta_h[censored], s_h[censored]))

    trace_hetero = pm.sample(1000, tune=2000, cores=4, chains=4,
                             random_seed=42, target_accept=0.9)

print(az.summary(trace_hetero, var_names=['alpha', 'rho']))

## 6. Log-Logistic AFT Model

The Weibull assumes a monotonic hazard rate. The **Log-Logistic** allows non-monotonic hazards (hump-shaped). In log-time, it uses a Logistic distribution instead of a Gumbel.

In [ ]:
def logistic_log_sf(y, mu, sigma):
    """Log survival function of the Logistic distribution."""
    return -pt.softplus((y - mu) / sigma)


with pm.Model() as loglogistic_aft:
    alpha_ll = pm.Normal('alpha', mu=0, sigma=2, shape=3)
    log_s_ll = pm.Normal('log_s', mu=0, sigma=1)
    s_ll = pm.Deterministic('s', pm.math.exp(log_s_ll))

    eta_ll = alpha_ll[0] + alpha_ll[1] * spend_std + alpha_ll[2] * tickets_std

    y_obs_ll = pm.Logistic('y_obs', mu=eta_ll[~censored], s=s_ll,
                            observed=log_observed_time[~censored])
    y_cens_ll = pm.Potential('y_cens',
        logistic_log_sf(log_observed_time[censored], eta_ll[censored], s_ll))

    trace_ll = pm.sample(1000, tune=2000, cores=4, chains=4,
                         random_seed=42, target_accept=0.9)

print(az.summary(trace_ll, var_names=['alpha', 's']))

### Compare Weibull vs Log-Logistic

In [ ]:
alpha_post_ll = trace_ll.posterior['alpha'].values.reshape(-1, 3)
s_post_ll = trace_ll.posterior['s'].values.flatten()


def compute_survival_logistic(alpha_post, s_post, sp, tk, log_t_grid):
    """Compute Log-Logistic survival curves from posterior samples."""
    eta_post = alpha_post[:, 0] + alpha_post[:, 1] * sp + alpha_post[:, 2] * tk
    survival = np.zeros((len(eta_post), len(log_t_grid)))
    for i in range(len(eta_post)):
        z = (log_t_grid - eta_post[i]) / s_post[i]
        survival[i] = 1 / (1 + np.exp(z))
    return survival


fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Weibull
ax = axes[0]
surv_w = compute_survival_weibull(alpha_post, s_post, 0.0, 0.0, log_t_grid)
ax.plot(t_grid, surv_w.mean(axis=0), color='#2196F3', lw=2.5, label='Weibull AFT')
ax.fill_between(t_grid, np.percentile(surv_w, 3, axis=0),
                np.percentile(surv_w, 97, axis=0), color='#2196F3', alpha=0.15)
ax.set_xlabel('Months since signup', fontsize=12)
ax.set_ylabel('Survival probability', fontsize=12)
ax.set_title('Weibull AFT', fontsize=14)
ax.set_xlim(0, 36)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)

# Log-Logistic
ax = axes[1]
surv_ll = compute_survival_logistic(alpha_post_ll, s_post_ll, 0.0, 0.0, log_t_grid)
ax.plot(t_grid, surv_ll.mean(axis=0), color='#FF9800', lw=2.5, label='Log-Logistic AFT')
ax.fill_between(t_grid, np.percentile(surv_ll, 3, axis=0),
                np.percentile(surv_ll, 97, axis=0), color='#FF9800', alpha=0.15)
ax.set_xlabel('Months since signup', fontsize=12)
ax.set_title('Log-Logistic AFT', fontsize=14)
ax.set_xlim(0, 36)
ax.legend(fontsize=10)

fig.suptitle('Average Customer: Weibull vs Log-Logistic', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Model Comparison with LOO-CV

In [ ]:
try:
    weibull_loo = az.loo(trace, weibull_aft)
    ll_loo = az.loo(trace_ll, loglogistic_aft)
    comparison = az.compare({'Weibull': trace, 'Log-Logistic': trace_ll})
    print(comparison)
except Exception as e:
    print(f"LOO comparison requires compute_log_likelihood. Error: {e}")
    print("Since our data was generated from a Weibull, the Weibull model should fit better.")

## 7. Survival Curve Convergence Animation

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

sample_counts = [5, 10, 25, 50, 100, 250, 500, 1000, 2000, 4000]
profile_data = [
    ('High-value', 1.5, -1.0, '#2196F3'),
    ('Average', 0.0, 0.0, '#FF9800'),
    ('At-risk', -1.5, 2.0, '#F44336'),
]

fig, ax = plt.subplots(figsize=(8, 5))

def update(frame):
    ax.clear()
    n = sample_counts[frame]
    for name, sp, tk, color in profile_data:
        eta_p = alpha_post[:n, 0] + alpha_post[:n, 1] * sp + alpha_post[:n, 2] * tk
        surv = np.zeros((n, len(t_grid)))
        for i in range(n):
            z = (log_t_grid - eta_p[i]) / s_post[:n][i]
            surv[i] = 1 - np.exp(-np.exp(-z))
        ax.plot(t_grid, surv.mean(axis=0), color=color, lw=2, label=name)
        if n >= 50:
            ax.fill_between(t_grid, np.percentile(surv, 3, axis=0),
                           np.percentile(surv, 97, axis=0), color=color, alpha=0.1)
    ax.set_xlabel('Months since signup')
    ax.set_ylabel('Survival probability')
    ax.set_title(f'Posterior samples: {n:,}')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(0, 36)
    ax.set_ylim(0, 1.05)

anim = FuncAnimation(fig, update, frames=len(sample_counts), interval=600)
plt.close(fig)
HTML(anim.to_jshtml())

## Exercises

1. **Different censoring rate**: Change `observation_window` to 12 or 36 months. How does the censoring fraction affect posterior uncertainty?

2. **More covariates**: Add a third covariate (e.g., `login_frequency`) to the data generation and model. Does it improve the fit?

3. **Hierarchical AFT**: Combine this post with the [hierarchical regression post](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/hierarchical_bayesian_regression.ipynb). Add a group index (e.g., subscription tier: Free, Pro, Enterprise) and make the `alpha` coefficients hierarchical. How does partial pooling affect survival curve estimates for the smallest group?

4. **Real data**: Replace the synthetic data with the `flchain` dataset from `statsmodels` (`sm.datasets.get_rdataset('flchain', 'survival')`). This is a real survival dataset with age, sex, and serum biomarkers. Fit a Weibull AFT model and interpret the coefficients.

5. **Model comparison**: Implement a log-Normal AFT model (use `pm.Normal` instead of `pm.Gumbel` for log-time) and compare it against the Weibull and Log-Logistic using LOO-CV.